In [0]:
import pandas as pd 
import os
import re
import pyspark
import datetime
import logging
import json
import requests
import urllib3
import time
from datetime import datetime
from pathlib import Path
from pyspark.sql import SparkSession
import configparser
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType
from pyspark.sql import functions as F

def upload_to_sql(metadata_sql_table: str, jsonFile: str, cross_bu_json_path: str, schema: StructType) -> pyspark.sql.dataframe.DataFrame :
        # jsonFile="WEBI_Document_139747.json" ### sample testing file

    buffer = []
    file_path = os.path.join(cross_bu_json_path, jsonFile)
    with open(file_path, "r") as file:
        data = json.load(file) if isinstance(json.load(open(file_path)), list) else [json.load(open(file_path))]
    webi_data=data[0].get("Document_Details",{})
    webi_data.update(data[0].get("Extraction_Stats",{}))

    for dataprovider in data[0].get("Data_Providers",[]):
        dataprovider_full={**webi_data, **dataprovider}; 
        dataprovider_full['scheduled'] = bool(dataprovider_full.get('scheduled', False))
        # print(dataprovider_full)
        buffer.append(dataprovider_full)
        
    
    cols = [f.name for f in schema.fields]
    rows = [{c: d.get(c, None) for c in cols} for d in buffer]
    batch_df = spark.createDataFrame(rows, schema=schema)
    buffer.clear()
    return batch_df
# webi_df = spark.createDataFrame([dataprovider_full], schema)

def main():
    spark = SparkSession.builder \
        .appName("Read JSON into DataFrame") \
        .getOrCreate()
    Todyay_Date = datetime.now()
    local_directory=os.getcwd()
    logs_path=os.path.join(local_directory,f"{local_directory}/logs")
    Path(logs_path).mkdir(parents=True, exist_ok=True)
    logs_path=os.path.join(logs_path, f"Script_Dev_{Todyay_Date}.log")

    logger = logging.getLogger("my_notebook_logger")
    logger.setLevel(logging.INFO)

    if logger.handlers:
        for h in list(logger.handlers):
            logger.removeHandler(h)

    fh = logging.FileHandler(logs_path, mode="a", encoding="utf-8")
    
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    logger.info("Notebook started.")
    # logger.warning("This is a test warning.")


    # logging.basicConfig(
    #     filename=f"Script_Dev_{Todyay_Date}.log",          # Dev Test Log file name
    #     # filename='Script_Prod_MB.log',          #Prod Log file name
    #     level=logging.INFO,          # Log level: DEBUG, INFO, WARNING, ERROR, CRITICAL
    #     format='%(asctime)s - %(levelname)s - %(message)s'
    # )
    # logging.info(f"Script started: {Todyay_Date} with directory {os.getcwd()}")  
    
    # print(os.listdir(logs))
    # cross_bu_json_path = f"{local_directory}/cross_bu_json_files"
    # within_bu=f"{local_directory}/within_bu_json_files"
    # private_bu=f"{local_directory}/private_json_files"
    # audit_additional=f"{local_directory}/audit_additional_json_files"
    non_active=f"{local_directory}/non_active-schedules"
    # print(os.listdir
    test_json=f"{local_directory}/test_json"
    metadata_sql_table="dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.webi_metadata_cms"

    schema = StructType([
        StructField("Document_CUID", StringType(), True),
        StructField("Document_Id", StringType(), True),
        StructField("Document_name", StringType(), True),
        StructField("Folder_Id", StringType(), True),
        StructField("Full_path", StringType(), True),
        StructField("created", StringType(), True),
        StructField("lastAuthor", StringType(), True),
        StructField("scheduled", BooleanType(), True),
        StructField("updated", StringType(), True),
        StructField("API_Pause_Counts", StringType(), True),
        StructField("End_Time", StringType(), True),
        StructField("Execution_Time_in_seconds", StringType(), True),
        StructField("Extraction_Stats", StringType(), True),
        StructField("Number_of_API_Calls", StringType(), True),
        StructField("Number_of_Data_Providers", StringType(), True),
        StructField("Start_Time", StringType(), True),
        StructField("Data_Provider_ID", StringType(), True),
        StructField("Data_Provider_Name", StringType(), True),
        StructField("DataSource_ID", StringType(), True),
        StructField("DataSource_CUID", StringType(), True),
        StructField("Data_Profider_Refresh_Time", StringType(), True),
        StructField("DataSource_Type", StringType(), True),
        StructField("DataSource_Name", StringType(), True),
        StructField("SQL_Index", StringType(), True),
        StructField("SQL_Query", StringType(), True),
        StructField("Connection_ID", StringType(), True),
        StructField("Connection_Name", StringType(), True),
        StructField("Connection_Type", StringType(), True),
        StructField("Connection_DataBas", StringType(), True),
        StructField("Connection_Network", StringType(), True),
    ])

    logger.info(f"Number of files to be processed: {len(os.listdir(non_active))}")
   
    ID_existed_df =spark.sql("select distinct Document_Id from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.webi_metadata_cms")
    logger.info(f"Number of WEBI loaded to SQL: {ID_existed_df.count()}")
    webi_df = spark.createDataFrame([], schema)
    # # # # ID_existed_df.show()
    countdown=30
    
    files = sorted(os.listdir(non_active))  # deterministic order
    n = len(files)

    for i, jsonFile in enumerate(files):
        try:
            numbers = re.findall(r"\d+", jsonFile)
            webi_id=numbers[0]
            # webi_id_1 = webi_id.strip().strip('"')
            rows = ID_existed_df.filter(F.col("Document_Id") == webi_id)
            # rows.show()             # preview  
            if  rows.count()>0:
                logger.info(f"WEBI {webi_id} already existed in SQL, skip")
                continue
            else:
                logger.info(f"{webi_id} to be processed")
                webi_df_1=upload_to_sql(metadata_sql_table, jsonFile, non_active, schema)
                webi_df = webi_df.unionByName(webi_df_1, allowMissingColumns=True)
                countdown=countdown-1
        except Exception as e:
            logger.error(f"Error: {e}")

        is_last = ( i == n - 1)
        if countdown == 0 or is_last:
            webi_df.write.format("delta").mode("append").insertInto(metadata_sql_table)
            if countdown == 0:
                countdown=30
                logger.info(f"{countdown} files processed, writing to SQL")
            else:
                logger.info(f"Last file {jsonFile} processed, writing to SQL")

        # webi_df.write.format("delta").mode("append").insertInto(metadata_sql_table)

main()

In [0]:
%sql


select distinct Document_Id from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.webi_metadata_cms
--  where Document_Id='179620'